# External English Comparison Using SemEval-2019 Hyperpartisan News

This notebook applies the MAGPIE article-level scoring pipeline to the SemEval-2019 Hyperpartisan News dataset. The dataset contains article-level labels indicating whether an article is hyperpartisan, providing an external English-language comparison for the MAGPIE analysis.

The same general scoring procedure used in the German and BABE analyses is followed here. Sentence-level MAGPIE predictions are aggregated to the article level using average, maximum, and median pooling. Statistical tests and article-length checks are then used to examine the resulting scores.

**Dataset:** SemEval-2019 Task 4: Hyperpartisan News Detection  
**Model:** `mediabiasgroup/magpie-babe-ft-xlm`

## 1. Setup

The required packages are installed first, followed by the imports used for the analysis. The same MAGPIE checkpoint is used as in the German and BABE notebooks so that the scoring procedure remains consistent across all three analyses.

Run the notebook **from top to bottom** after uploading the four SemEval XML files.

In [1]:
!pip install transformers pandas scipy statsmodels tqdm -q


## 2. Load and combine the data

The SemEval-2019 by-article dataset is provided through separate XML files containing the article text and hyperpartisan labels for the training and test sets. These files are parsed and then merged using the article ID.

The analysis uses the manually labelled by-article data rather than the larger distant-supervision dataset. The resulting dataset contains the article text together with a binary `hyperpartisan` label.

In [2]:
import xml.etree.ElementTree as ET
import pandas as pd

tree = ET.parse("articles-training-byarticle-20181122.xml")
root = tree.getroot()

rows = []
for article in root.findall("article"):
    rows.append({
        "id": article.get("id"),
        "title": article.get("title"),
        "published_at": article.get("published-at"),
        "text": "".join(article.itertext()).strip()
    })

train_articles_df = pd.DataFrame(rows)
print("train_articles_df:", train_articles_df.shape)

train_articles_df: (645, 4)


In [3]:
tree = ET.parse("ground-truth-training-byarticle-20181122 2.xml")
root = tree.getroot()

rows = []
for article in root.findall("article"):
    rows.append({
        "id": article.get("id"),
        "hyperpartisan": article.get("hyperpartisan")
    })

train_labels_df = pd.DataFrame(rows)
print("train_labels_df:", train_labels_df.shape)

train_labels_df: (645, 2)


In [4]:
tree = ET.parse("articles-test-byarticle-20181207.xml")
root = tree.getroot()

rows = []
for article in root.findall("article"):
    rows.append({
        "id": article.get("id"),
        "title": article.get("title"),
        "published_at": article.get("published-at"),
        "text": "".join(article.itertext()).strip()
    })

test_articles_df = pd.DataFrame(rows)
print("test_articles_df:", test_articles_df.shape)

test_articles_df: (628, 4)


In [5]:
tree = ET.parse("ground-truth-test-byarticle-20181207.xml")
root = tree.getroot()

rows = []
for article in root.findall("article"):
    rows.append({
        "id": article.get("id"),
        "hyperpartisan": article.get("hyperpartisan")
    })

test_labels_df = pd.DataFrame(rows)
print("test_labels_df:", test_labels_df.shape)

test_labels_df: (628, 2)


## 2.1 Data quality checks

Before the datasets are combined, a few basic data quality checks are performed. Article IDs are checked for duplicates, article text is checked for missing or empty values, and the label files are checked to ensure that the required labels are available for the articles included in the analysis.

In [6]:
# Check the parsed article and label files before merging.

print("Training articles:", len(train_articles_df))
print("Training labels:", len(train_labels_df))
print("Test articles:", len(test_articles_df))
print("Test labels:", len(test_labels_df))

print("\nMissing training text:", train_articles_df["text"].isna().sum())
print("Empty training text:", train_articles_df["text"].str.strip().eq("").sum())
print("Duplicate training article IDs:", train_articles_df["id"].duplicated().sum())
print("Duplicate training label IDs:", train_labels_df["id"].duplicated().sum())

print("\nMissing test text:", test_articles_df["text"].isna().sum())
print("Empty test text:", test_articles_df["text"].str.strip().eq("").sum())
print("Duplicate test article IDs:", test_articles_df["id"].duplicated().sum())
print("Duplicate test label IDs:", test_labels_df["id"].duplicated().sum())

assert train_articles_df["id"].notna().all()
assert train_labels_df["id"].notna().all()
assert test_articles_df["id"].notna().all()
assert test_labels_df["id"].notna().all()
assert train_articles_df["text"].str.strip().ne("").all()
assert test_articles_df["text"].str.strip().ne("").all()
assert train_articles_df["id"].is_unique
assert train_labels_df["id"].is_unique
assert test_articles_df["id"].is_unique
assert test_labels_df["id"].is_unique


Training articles: 645
Training labels: 645
Test articles: 628
Test labels: 628

Missing training text: 0
Empty training text: 0
Duplicate training article IDs: 0
Duplicate training label IDs: 0

Missing test text: 0
Empty test text: 0
Duplicate test article IDs: 0
Duplicate test label IDs: 0


In [7]:
train_full = train_articles_df.merge(train_labels_df, on="id")
test_full = test_articles_df.merge(test_labels_df, on="id")

semeval_data = pd.concat([train_full, test_full], ignore_index=True)

print("Final combined dataset:", semeval_data.shape)
semeval_data["hyperpartisan"].value_counts()

Final combined dataset: (1273, 5)


,count
hyperpartisan,
false,721
true,552


## 3. Score articles with MAGPIE

The same article-level scoring procedure used in the German and BABE analyses is applied here. Each article is split into sentences, and MAGPIE is used to obtain a bias probability for each sentence. These sentence-level probabilities are then aggregated within each article using average, maximum, and median pooling.

In [8]:
from transformers import pipeline
import re

magpie_classifier = pipeline(
    "text-classification",
    device=0,  # Use the GPU when available for sentence-level scoring.
    model="mediabiasgroup/magpie-babe-ft"
)

def split_into_sentences(text):
    # Split articles into sentences using the same rule as the other notebooks.
    sentences = re.split(r'(?<=[.!?]) +', text)
    return [s.strip() for s in sentences if len(s.strip()) > 0]

config.json:   0%|          | 0.00/835 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/451 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

In [9]:
import statistics

def get_article_scores(text, classifier, batch_size=32):
    """Scores every sentence in an article once, then returns all three
    pooling values from that single pass - no reason to re-run the model
    separately for each pooling method."""
    if pd.isna(text) or len(text.strip()) == 0:
        return None, None, None

    sentences = split_into_sentences(text)
    if len(sentences) == 0:
        return None, None, None

    results = classifier(sentences, truncation=True, batch_size=batch_size)

    biased_probs = []
    for r in results:
        if r["label"] == "biased":
            biased_probs.append(r["score"])
        else:
            biased_probs.append(1 - r["score"])

    avg_score = sum(biased_probs) / len(biased_probs)
    max_score = max(biased_probs)
    median_score = statistics.median(biased_probs)

    return avg_score, max_score, median_score


In [10]:
from tqdm import tqdm

avg_scores = []
max_scores = []
median_scores = []

for idx, row in tqdm(semeval_data.iterrows(), total=len(semeval_data)):
    avg_s, max_s, median_s = get_article_scores(row["text"], magpie_classifier)
    avg_scores.append(avg_s)
    max_scores.append(max_s)
    median_scores.append(median_s)

semeval_data["avg_bias_score"] = avg_scores
semeval_data["max_bias_score"] = max_scores
semeval_data["median_bias_score"] = median_scores

print("Done!")
print(f"Articles with no score: {semeval_data['avg_bias_score'].isna().sum()}")
semeval_data.head()

100%|██████████| 1273/1273 [00:56<00:00, 22.65it/s]

Done!
Articles with no score: 0


,id,title,published_at,text,hyperpartisan,avg_bias_score,max_bias_score,median_bias_score
0,0000000,Kucinich: Reclaiming the money power,2017-09-10,From flickr.com: Money {MID-161793} Money ( Im...,true,0.257701,0.949349,0.115615
1,0000001,Trump Just Woke Up & Viciously Attacked Puerto...,2017-10-12,Donald Trump ran on many braggadocios and larg...,true,0.532978,0.978996,0.500627
2,0000002,"Liberals wailing about gun control, but what a...",2017-10-11,Photo By Justin Sullivan/Getty Images In respo...,true,0.486812,0.971568,0.436156
3,0000003,Laremy Tunsil joins NFL players in kneeling du...,2017-09-24,After Colin Kaepernick rightly chose to kneel ...,true,0.524688,0.980061,0.510899
4,0000004,It's 1968 All Over Again,2017-10-12,"Almost a half-century ago, in 1968, the United...",false,0.582434,0.982433,0.788849


## 4. Statistical comparison

The three article-level MAGPIE scores are compared between hyperpartisan and non-hyperpartisan articles using Mann–Whitney U tests. The aim is to examine whether MAGPIE scores differ systematically between the two labelled groups in this external English-language dataset.

In [11]:
from scipy.stats import mannwhitneyu

true_avg = semeval_data[semeval_data["hyperpartisan"] == "true"]["avg_bias_score"]
false_avg = semeval_data[semeval_data["hyperpartisan"] == "false"]["avg_bias_score"]

true_max = semeval_data[semeval_data["hyperpartisan"] == "true"]["max_bias_score"]
false_max = semeval_data[semeval_data["hyperpartisan"] == "false"]["max_bias_score"]

true_median = semeval_data[semeval_data["hyperpartisan"] == "true"]["median_bias_score"]
false_median = semeval_data[semeval_data["hyperpartisan"] == "false"]["median_bias_score"]

stat_avg, p_avg = mannwhitneyu(true_avg, false_avg, alternative="two-sided")
stat_max, p_max = mannwhitneyu(true_max, false_max, alternative="two-sided")
stat_median, p_median = mannwhitneyu(true_median, false_median, alternative="two-sided")

print("AVERAGE POOLING")
print(f"  Hyperpartisan=true mean: {true_avg.mean():.4f}")
print(f"  Hyperpartisan=false mean: {false_avg.mean():.4f}")
print(f"  P-value: {p_avg:.4f}")
print()
print("MAX POOLING")
print(f"  Hyperpartisan=true mean: {true_max.mean():.4f}")
print(f"  Hyperpartisan=false mean: {false_max.mean():.4f}")
print(f"  P-value: {p_max:.4f}")
print()
print("MEDIAN POOLING")
print(f"  Hyperpartisan=true median: {true_median.mean():.4f}")
print(f"  Hyperpartisan=false median: {false_median.mean():.4f}")
print(f"  P-value: {p_median:.4f}")

def rank_biserial_from_u(u, n1, n2):
    return (2 * u) / (n1 * n2) - 1

rb_avg = rank_biserial_from_u(stat_avg, len(true_avg), len(false_avg))
rb_max = rank_biserial_from_u(stat_max, len(true_max), len(false_max))
rb_median = rank_biserial_from_u(stat_median, len(true_median), len(false_median))

alpha_bonferroni = 0.05 / 3

print()
print(f"Rank-biserial effect size (average): {rb_avg:.4f}")
print(f"Rank-biserial effect size (max): {rb_max:.4f}")
print(f"Rank-biserial effect size (median): {rb_median:.4f}")
print(f"Bonferroni-corrected alpha: {alpha_bonferroni:.4f}")


AVERAGE POOLING
  Hyperpartisan=true mean: 0.5122
  Hyperpartisan=false mean: 0.2937
  P-value: 0.0000

MAX POOLING
  Hyperpartisan=true mean: 0.9604
  Hyperpartisan=false mean: 0.7884
  P-value: 0.0000

MEDIAN POOLING
  Hyperpartisan=true median: 0.5169
  Hyperpartisan=false median: 0.2161
  P-value: 0.0000

Rank-biserial effect size (average): 0.6502
Rank-biserial effect size (max): 0.7036
Rank-biserial effect size (median): 0.6286
Bonferroni-corrected alpha: 0.0167


## 5. Checking for a length confound

Article length is considered because the number of sentences in an article may affect the pooled MAGPIE scores. The analysis first examines the relationship between sentence count and the three article-level scores. Regression models are then used to check whether the difference between hyperpartisan and non-hyperpartisan articles remains after accounting for article length.

In [12]:
# checking if hyperpartisan articles are systematically longer or shorter than non-hyperpartisan ones
semeval_data["num_sentences"] = semeval_data["text"].apply(lambda t: len(split_into_sentences(t)))

print(semeval_data.groupby("hyperpartisan")["num_sentences"].describe())
print()
print(semeval_data[["num_sentences", "avg_bias_score", "max_bias_score", "median_bias_score"]].corr())

               count       mean        std  min   25%   50%    75%    max
hyperpartisan                                                            
false          721.0  20.883495  21.172906  1.0   9.0  15.0  24.00  196.0
true           552.0  36.864130  27.844291  1.0  19.0  31.0  46.25  234.0

                   num_sentences  avg_bias_score  max_bias_score  \
num_sentences           1.000000        0.150213        0.305503   
avg_bias_score          0.150213        1.000000        0.616518   
max_bias_score          0.305503        0.616518        1.000000   
median_bias_score       0.119289        0.938260        0.426051   

                   median_bias_score  
num_sentences               0.119289  
avg_bias_score              0.938260  
max_bias_score              0.426051  
median_bias_score           1.000000  


In [13]:
import statsmodels.formula.api as smf

# Encode the binary label for the regression models.
semeval_data["hyperpartisan_numeric"] = (semeval_data["hyperpartisan"] == "true").astype(int)

print("AVERAGE SCORE, controlling for length:")
model_avg = smf.ols("avg_bias_score ~ hyperpartisan_numeric + num_sentences", data=semeval_data).fit(cov_type="HC3")
print(model_avg.summary().tables[1])

print()
print("MAX SCORE, controlling for length:")
model_max = smf.ols("max_bias_score ~ hyperpartisan_numeric + num_sentences", data=semeval_data).fit(cov_type="HC3")
print(model_max.summary().tables[1])

print()
print("MEDIAN SCORE, controlling for length:")
model_median = smf.ols("median_bias_score ~ hyperpartisan_numeric + num_sentences", data=semeval_data).fit(cov_type="HC3")
print(model_median.summary().tables[1])

AVERAGE SCORE, controlling for length:
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                 0.2972      0.008     38.057      0.000       0.282       0.313
hyperpartisan_numeric     0.2212      0.010     22.515      0.000       0.202       0.240
num_sentences            -0.0002      0.000     -0.906      0.365      -0.001       0.000

MAX SCORE, controlling for length:
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                 0.7494      0.012     61.641      0.000       0.726       0.773
hyperpartisan_numeric     0.1422      0.010     14.287      0.000       0.123       0.162
num_sentences             0.0019      0.000      8.483      0.000       0.001       0.002

MEDIAN SCORE, controllin

## 6. Interpretation

The SemEval analysis provides an external English-language comparison of the article-level MAGPIE pipeline. The results are considered alongside the BABE sanity check and the German analysis to examine whether similar article-level patterns are observed across datasets.

The SemEval labels and the German political-bias labels do not represent exactly the same construct. Therefore, the results are treated as comparative evidence about the behaviour of the MAGPIE scoring pipeline rather than as definitive validation of political bias detection.